In [1]:
import re
import json
import requests
from bs4 import BeautifulSoup
import pandas as pd
from requests.exceptions import ConnectionError, Timeout, HTTPError

def extract_json_variable(html, variable_name):
    if not html:
        return None
    soup = BeautifulSoup(html, 'html.parser')
    script_tag = soup.find("script", string=re.compile(fr"{variable_name}\s*="))
    if script_tag:
        match = re.search(fr"{variable_name}\s*=\s*({{.*?}});", script_tag.string, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                return None
    return None

def fetch_html(url):
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        response.raise_for_status()
        return response.text
    except (ConnectionError, Timeout, HTTPError) as e:
        print(f"Skipping URL due to error: {url} - {e}")
        return None

def read_urls_from_file(file_path):
    with open(file_path, 'r') as file:
        urls = file.readlines()
    return [url.strip() for url in urls]

def process_urls(urls):
    all_data = []
    for url in urls:
        print(f"Processing URL: {url}")
        html = fetch_html(url)
        if not html:
            continue  # Skip to the next URL if fetching failed

        initial_state = extract_json_variable(html, "window.__INITIAL_STATE__")
        if not initial_state:
            print(f"No valid product data found for {url}, skipping...")
            continue
        
        try:
            for key, value in initial_state.items():
                if isinstance(value, dict) and "product" in value:
                    product_data = value["product"]
                    data = {
                        'Category': 'Appliances',
                        "URL": url,
                        "sku": product_data.get('sku', "N/A"),
                        "Product Name": product_data.get("name", "N/A"),
                        "Brand": product_data.get("brandName", "N/A"),
                        "Regular Price": product_data.get("regularPrice", "N/A"),
                        "Discount": product_data.get("saving", "N/A"),
                        "Discounted Price": product_data.get("priceWithEhf", "N/A"),
                        "Is Clearance": product_data.get("isClearance", "N/A"),
                        "On Sale": product_data.get("isOnSale", "N/A"),
                        "Sale Start Date": product_data.get("saleStartDate", "N/A"),
                        "Sale End Date": product_data.get("saleEndDate", "N/A"),
                        "Customer Rating": product_data.get("customerRating", "N/A"),
                        "Total Reviews": product_data.get("customerRatingCount", "N/A"),
                    }
                    all_data.append(data)
                    break  # Exit loop after processing the first product
        except KeyError as e:
            print(f"Error extracting product details for {url}: {e}")
    
    return pd.DataFrame(all_data)

file_path = 'C:/Users/91892/Downloads/captured_product_urls.txt'
urls = read_urls_from_file(file_path)
product_df = process_urls(urls)
print(product_df)


Processing URL: https://www.bestbuy.ca/en-ca/product/dyson-v7-advanced-cordless-stick-vacuum-silver/17984714
Processing URL: https://www.bestbuy.ca/en-ca/product/irobot-roomba-j7-wi-fi-connected-self-empty-robot-vacuum-j7550/15728794
Processing URL: https://www.bestbuy.ca/en-ca/product/dyson-v15-detect-cordless-stick-vacuum-yellow-nickel/17183862
Processing URL: https://www.bestbuy.ca/en-ca/product/vitamix-e320-1-89l-1500-watt-stand-blender-black-only-at-best-buy/18192441
Processing URL: https://www.bestbuy.ca/en-ca/product/philips-2000-series-air-fryer-with-window-6-6qt-black-only-at-best-buy/17916312
Processing URL: https://www.bestbuy.ca/en-ca/product/galanz-expresswave-1-3-cu-ft-microwave-gewwd13s5sv11-stainless-steel/17853017
Processing URL: https://www.bestbuy.ca/en-ca/product/bobsweep-dustin-wifi-connected-self-empty-robot-vacuum-mop-night-black/18246162
Processing URL: https://www.bestbuy.ca/en-ca/product/dyson-v11-cordless-stick-vacuum-nickel-blue/17183829
Processing URL: http

In [2]:
product_df

,Category,URL,sku,Product Name,Brand,Regular Price,Discount,Discounted Price,Is Clearance,On Sale,Sale Start Date,Sale End Date,Customer Rating,Total Reviews
0,Appliances,https://www.bestbuy.ca/en-ca/product/dyson-v7-...,17984714,Dyson V7 Advanced Cordless Stick Vacuum - Silver,DYSON,499.99,150.0,349.99,False,True,2025-01-17T08:00:00Z,2025-01-30T23:59:59-06:00,3.98,391
1,Appliances,https://www.bestbuy.ca/en-ca/product/irobot-ro...,15728794,iRobot Roomba j7+ Wi-Fi Connected Self-Empty R...,IROBOT,999.99,520.0,479.99,False,True,2025-01-10T08:00:00Z,2025-01-31T23:59:59-06:00,4.26,1846
2,Appliances,https://www.bestbuy.ca/en-ca/product/dyson-v15...,17183862,Dyson V15 Detect Cordless Stick Vacuum - Yello...,DYSON,999.99,200.0,799.99,False,True,2025-01-17T08:00:00Z,2025-01-30T23:59:59-06:00,4.60,5567
3,Appliances,https://www.bestbuy.ca/en-ca/product/vitamix-e...,18192441,Vitamix E320 1.89L 1500-Watt Stand Blender - B...,VITAMIX,419.99,70.0,349.99,False,True,2025-01-17T08:00:00Z,2025-01-30T23:59:59-06:00,4.32,53
4,Appliances,https://www.bestbuy.ca/en-ca/product/philips-2...,17916312,Philips 2000 Series Air Fryer with Window - 6....,PHILIPS,199.99,0.0,199.99,False,False,2025-01-24T08:00:00Z,2025-01-31T00:00:00-06:00,4.85,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2246,Appliances,https://www.bestbuy.ca/en-ca/product/insignia-...,17543707,Insignia 1.1 Cu. Ft. Countertop Microwave (NS-...,INSIGNIA,149.99,50.0,99.99,False,True,2025-01-24T08:00:00Z,2025-01-30T23:59:59-06:00,4.72,265
2247,Appliances,https://www.bestbuy.ca/en-ca/product/instant-p...,16609677,Instant Pot Duo Plus 9-in-1 Electric Pressure ...,INSTANT POT,199.99,0.0,199.99,False,False,2025-01-24T08:00:00Z,2025-01-31T00:00:00-06:00,4.73,611
2248,Appliances,https://www.bestbuy.ca/en-ca/product/narwal-fr...,17857430,Narwal Freo X Ultra Cordless Robot Vacuum & Mo...,NARWAL,1499.99,200.0,1299.99,False,True,2025-01-25T08:00:00Z,2025-01-30T23:59:59-06:00,4.96,26
2249,Appliances,https://www.bestbuy.ca/en-ca/product/delonghi-...,16250764,DeLonghi Magnifica Evo Automatic Espresso Make...,DE'LONGHI,999.99,0.0,999.99,False,False,2025-01-24T08:00:00Z,2025-01-31T00:00:00-06:00,4.60,1012


In [4]:
product_df.to_csv('C:/Users/91892/Downloads/output2kappliances.csv', index=True)  # Set index=False to exclude row indices

print("CSV file saved successfully!")

CSV file saved successfully!
